# 六状态静止直立 LQR

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'base.py').exists():
    if root==root.parent: raise RuntimeError('找不到 theory/base.py')
    root=root.parent
sys.path.insert(0,str(root))
import sympy as s
import numpy as np
from IPython.display import display, Math
# 只改 LaTeX 显示，不替换计算符号：dx -> x 上点，ddx -> x 双点。
from base import q, qd, qdd
latex_names = {}
for coordinate, velocity, acceleration in zip(q, qd, qdd):
    name = s.latex(coordinate)
    latex_names[velocity] = r'\dot{' + name + '}'
    latex_names[acceleration] = r'\ddot{' + name + '}'
s.init_printing(symbol_names=latex_names)
np.set_printoptions(precision=8,suppress=True,linewidth=130)
def show(label, expression):
    display(Math(label+' = '+s.latex(expression, symbol_names=latex_names)))


In [2]:
from straight_balance.lqr import *
result=design()

## 状态方程
$$X=[\delta x,\delta\psi,\delta\theta,\delta\dot x,\delta\dot\psi,\delta\dot\theta]^T,$$
$$\dot X=AX+B\tau,\quad A=\begin{bmatrix}0&I\\-M_0^{-1}K_0&-M_0^{-1}C_0\end{bmatrix},\quad B=\begin{bmatrix}0\\M_0^{-1}B_0\end{bmatrix}.$$
下面先展示符号矩阵，再代入当前项目的临时示例参数。数值实现使用 solve，不显式求逆。

In [3]:
As,Bs=symbolic();show('A',As);show('B',Bs)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 从性能指标到最终增益
下列 $E$ 表示所选线性误差状态（静止分支就是 $X$，转向分支是旋转坐标中的七维误差）。

$$\dot E=A_eE+B_e\delta\tau,\qquad
J=\int_0^\infty(E^TQE+\delta\tau^TR\delta\tau)\,dt.$$

$Q\succeq0$ 惩罚各状态偏差，$R\succ0$ 惩罚电机力矩。权重有量纲依赖，不是车辆物理参数。
在可稳定、可检测的线性问题上，设最小剩余代价 $V(E)=E^TPE$。

$$0=\min_{\delta\tau}\{E^TQE+\delta\tau^TR\delta\tau+2E^TP(A_eE+B_e\delta\tau)\}.$$

对力矩求导：$2R\delta\tau+2B_e^TPE=0$，故

$$\delta\tau_*=-R^{-1}B_e^TPE=-K_e E.$$

代回并比较二次型系数，得到 CARE：

$$A_e^TP+PA_e-PB_eR^{-1}B_e^TP+Q=0.$$

选择使 $A_e-B_eK_e$ 稳定的对称解 $P$，于是 $J_{\min}(E_0)=E_0^TPE_0$。
这给出最优反馈，而不是把积分式对常矩阵 $K$ 的每个元素蛮力求极值。
实际程序用 `solve_continuous_are` 求 $P$，用 `solve(R, B.T@P)` 得到 $K$。
下方展示 $Q,R,P,K$ 的实际数值及残差。该最优性只针对所写的线性误差系统和无饱和二次代价。

参考：[MIT Underactuated Robotics — LQR](https://underactuated.mit.edu/lqr.html)。


In [4]:
from parameters import values
print('临时示例参数（不是实车值）：',values()[0])
for key in ('A','B','Q','R','P','K'):show(key,s.Matrix(result[key]).evalf(8))
print('可控性秩：',result['rank'])
print('闭环特征值：',result['poles'])
print('CARE残差：',result['care_residual'])

临时示例参数（不是实车值）： {'r': 0.1, 'd': 0.4, 'l': 0.3, 'm_B': 2.0, 'm_L': 0.5, 'm_R': 0.5, 'I_Lyy': 0.0025, 'I_Ryy': 0.0025, 'I_Lzz': 0.00125, 'I_Rzz': 0.00125, 'I_Byy': 0.08, 'I_Bzz': 0.06, 'g': 9.81, 'psi0': 0.0, 'I_Bxx': 0.1, 'I_Lxx': 0.00125, 'I_Rxx': 0.00125}


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

可控性秩： 6
闭环特征值： [-49.84821689+0.j          -3.19941349+0.j          -0.76775527+0.59960155j  -0.76775527-0.59960155j -10.20101471+0.j
  -1.60048104+0.j        ]
CARE残差： 4.885781308099527e-12


## 最优代价示例
只给初始俯仰误差 0.03 rad，其他误差为0；下面得到该线性无限时域问题的最小代价数值。

In [5]:
E0=np.zeros(len(result['A']));E0[3 if len(E0)==7 else 2]=.03
show('J_{min}',s.Float(E0@result['P']@E0))

<IPython.core.display.Math object>